# Business Analytics Report — Task 2
### Drivers of Private Hospital Insurance Uptake in Australia

**Student:** Paula Moreno
**Topic:** Identifying the personal and household characteristics that influence the probability of holding Private Hospital Insurance (PHI) in Australia, using a Linear Probability Model (LPM).
**Data source:** PHI_2026.xlsx — a randomly generated individual-level sample (1,705 observations)

This notebook estimates a **Linear Probability Model (LPM)** of PHI uptake at the
individual level. Whereas Task 1 examined aggregate trends in PHI coverage over time,
this analysis focuses on the **micro-level determinants** of an individual's decision
to hold private hospital cover.

The notebook is organised into six sections:

1. **Section 1 — Data Exploration & Preparation.** Loads the dataset and constructs
   the age-squared term required by the model specification.
2. **Section 2 — Linear Probability Model.** Estimates the regression and reports
   the coefficient on each variable.
3. **Section 3 — Missing Variables.** Discusses important determinants of PHI uptake
   that are not observed in the dataset.
4. **Section 4 — Omitted Variable Bias & Reverse Causality.** Identifies potential
   threats to the causal interpretation of the estimated coefficients.
5. **Section 5 — Predicted Probabilities for a Synthetic Individual.** Uses the
   estimated model to predict PHI probability for a typical individual and shows
   how that probability changes with key variables.
6. **Section 6 — The Medicare Levy Surcharge.** Discusses how to estimate the
   causal effect of the MLS on PHI uptake using this dataset.

---
## Section 1 — Data Exploration and Preparation

This section loads the individual-level PHI dataset, performs a brief inspection
of the variables used in the model, and creates the age-squared term needed to
capture the non-linear relationship between age and PHI uptake identified in Task 1.

### 1.0 Environment Setup

Imports the libraries used in this notebook. In addition to the standard pandas and
visualisation libraries used in Task 1, this notebook uses **statsmodels** to estimate
the linear regression model with robust standard errors.

In [2]:
# Core data libraries
import pandas as pd
import numpy as np

# Regression library
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Consistent chart style
sns.set_theme(style="white", rc={"figure.figsize": (12, 6)})
sns.set_context("notebook", font_scale=1.15, rc={"lines.linewidth": 2.5})

In [3]:
from google.colab import files
uploaded = files.upload()

Saving PHI_2026-1.xlsx to PHI_2026-1.xlsx


### 1.1 Loading the Dataset

The PHI_2026 dataset is a simulated individual-level sample representing Australian
adults. It contains 1,705 observations and is designed to reflect realistic patterns
in the Australian Private Health Insurance market. The file contains two sheets:
`Variable Description` (a data dictionary) and `Dataset` (the actual data).

This section loads the data, briefly inspects its structure, and confirms that no
data cleaning is required before estimation.

In [4]:
# Load the individual-level dataset
phi_ind = pd.read_excel('PHI_2026-1.xlsx', sheet_name='Dataset')

# Quick inspection
print("Shape:", phi_ind.shape)
print("\nFirst 5 rows:")
phi_ind.head()

Shape: (1705, 12)

First 5 rows:


,personid,PHI,HHincome,employed,bachelor,depchildren,couple,age,female,health,city,life_sat
0,1,0,19,0,1,1,0,57,1,3,0,9
1,2,1,213,1,0,3,0,48,1,3,1,6
2,3,1,213,1,0,3,0,48,1,3,1,6
3,4,1,170,1,1,1,0,60,1,3,1,7
4,5,0,86,0,0,1,0,79,0,4,0,8


In [5]:
# Confirm data quality — no missing values, sensible ranges
print("=== Missing values per variable ===")
print(phi_ind.isna().sum())

print("\n=== Summary statistics ===")
phi_ind.describe().round(2)

=== Missing values per variable ===
personid       0
PHI            0
HHincome       0
employed       0
bachelor       0
depchildren    0
couple         0
age            0
female         0
health         0
city           0
life_sat       0
dtype: int64

=== Summary statistics ===


,personid,PHI,HHincome,employed,bachelor,depchildren,couple,age,female,health,city,life_sat
count,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000,1705.0000
mean,853.0000,0.5200,120.9400,0.6200,0.3000,1.6000,0.6600,53.0900,0.5800,2.6900,0.6500,7.9700
std,492.3400,0.5000,65.1300,0.4900,0.4600,0.9600,0.4700,15.9200,0.4900,0.9700,0.4800,1.3800
min,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,24.0000,0.0000,1.0000,0.0000,1.0000
25%,427.0000,0.0000,71.0000,0.0000,0.0000,1.0000,0.0000,41.0000,0.0000,2.0000,0.0000,7.0000
50%,853.0000,1.0000,115.0000,1.0000,0.0000,1.0000,1.0000,53.0000,1.0000,3.0000,1.0000,8.0000
75%,1279.0000,1.0000,170.0000,1.0000,1.0000,2.0000,1.0000,64.0000,1.0000,3.0000,1.0000,9.0000
max,1705.0000,1.0000,374.0000,1.0000,1.0000,4.0000,1.0000,99.0000,1.0000,5.0000,1.0000,10.0000


**Data quality note.** The dataset is complete (no missing values) and all variables
fall within their expected ranges as defined in the variable description sheet. No
data cleaning is required prior to estimation.

**Key sample features:**
- 1,705 individuals
- Overall PHI uptake rate: ~52% (consistent with the national average observed in Task 1)
- Ages range from 24 to 99 (mean ~53)
- Household income ranges from $0 to $374,000 (mean ~$121,000)
- Self-rated health is on a 1–5 scale where 1 = excellent and 5 = poor

### 1.2 Creating the Age-Squared Term

A central feature of the model specification is the inclusion of an **age-squared
term**. This is motivated both by economic theory and by the empirical evidence
from Task 1, which showed a clearly non-linear relationship between age and PHI
coverage: uptake rises with age through middle adulthood, peaks among older
pre-retirement Australians, and then declines among the oldest cohorts.

A purely linear age term would impose a monotonic relationship on the model,
incorrectly forcing the estimated effect of age to be either strictly positive or
strictly negative across the full range. Including `age²` allows the relationship
to bend, capturing the inverted-U pattern observed in the aggregate data.

This term is constructed as a new column equal to the square of `age` and added
directly to the dataset. No other transformations of the original variables are
required prior to estimation.

In [6]:
# Create the age-squared variable
phi_ind['age_sq'] = phi_ind['age'] ** 2

# Confirm it was created correctly
print("First 5 rows showing age and age_sq:")
print(phi_ind[['age', 'age_sq']].head())

print("\nSummary statistics for age_sq:")
print(phi_ind['age_sq'].describe().round(2))

First 5 rows showing age and age_sq:
   age  age_sq
0   57    3249
1   48    2304
2   48    2304
3   60    3600
4   79    6241

Summary statistics for age_sq:
count   1705.0000
mean    3072.0500
std     1775.9400
min      576.0000
25%     1681.0000
50%     2809.0000
75%     4096.0000
max     9801.0000
Name: age_sq, dtype: float64


**Interpretation of the new variable.** The `age_sq` column contains the square of
each individual's age. For example, a 24-year-old has `age_sq = 576`, while a
99-year-old has `age_sq = 9801`. The model will estimate two coefficients on age:
one on the linear term (`age`) and one on the squared term (`age_sq`). Together
they describe the shape of the age–PHI relationship and identify the age at which
uptake reaches its maximum.

### 2.1 The Equation of the Population Regression Model

The model is specified as follows:

$$
PHI_i = \beta_0 + \beta_1 HHincome_i + \beta_2 age_i + \beta_3 age^2_i + \beta_4 employed_i + \beta_5 bachelor_i + \beta_6 depchildren_i + \beta_7 couple_i + \beta_8 female_i + \beta_9 health_i + \beta_{10} city_i + u_i
$$

where:
- $PHI_i$ is a binary indicator equal to 1 if individual $i$ holds Private Hospital
  Insurance and 0 otherwise.
- $\beta_0$ is the intercept (the baseline probability when all variables equal zero).
- $\beta_1, ..., \beta_{10}$ are the parameters to be estimated.
- $u_i$ is the error term capturing unobserved factors that influence PHI uptake.

The model includes ten explanatory variables motivated by the economic theory
of health insurance demand and the existing empirical literature on the
Australian PHI market. Age is included in quadratic form (`age` and `age²`)
to allow the model to capture potential non-linearity in the age–uptake
relationship.

Because the dependent variable is binary, the model is estimated by OLS as a
**Linear Probability Model**. To address the heteroskedasticity inherent in
LPM estimation, **heteroskedasticity-robust (HC1) standard errors** are used
for all reported significance tests.

### 2.2 Estimation Results

The model is estimated below using the `statsmodels` package in Python. The
output reports the estimated coefficient for each variable, its robust
standard error, t-statistic, p-value, and the 95% confidence interval.
Statistical significance is assessed at the 10% level, as specified in the
assessment brief.

In [7]:
# Define the explanatory variables (in the order they appear in the equation)
X_vars = ['HHincome', 'age', 'age_sq',
          'employed', 'bachelor', 'depchildren',
          'couple', 'female', 'health', 'city']

# Build the design matrix (X) and dependent variable (y)
X = phi_ind[X_vars]
X = sm.add_constant(X)          # adds the intercept (β₀)
y = phi_ind['PHI']

# Estimate the Linear Probability Model with robust (HC1) standard errors
lpm = sm.OLS(y, X).fit(cov_type='HC1')

# Print the summary
print(lpm.summary())

                            OLS Regression Results                            
Dep. Variable:                    PHI   R-squared:                       0.209
Model:                            OLS   Adj. R-squared:                  0.204
Method:                 Least Squares   F-statistic:                     64.91
Date:                Tue, 09 Jun 2026   Prob (F-statistic):          6.24e-112
Time:                        00:35:45   Log-Likelihood:                -1036.9
No. Observations:                1705   AIC:                             2096.
Df Residuals:                    1694   BIC:                             2156.
Df Model:                          10                                         
Covariance Type:                  HC1                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0763      0.110     -0.694      

In [8]:
# === LPM Results Table — clean academic style ===

# Build the results dataframe
results_df = pd.DataFrame({
    'Variable':       lpm.params.index,
    'Coefficient':    lpm.params.values,
    'Std. Error':     lpm.bse.values,
    't-statistic':    lpm.tvalues.values,
    'p-value':        lpm.pvalues.values,
    '95% CI Lower':   lpm.conf_int()[0].values,
    '95% CI Upper':   lpm.conf_int()[1].values
})

# Display names
display_names = {
    'const':       'Intercept (β₀)',
    'HHincome':    'HHincome',
    'age':         'age',
    'age_sq':      'age_sq',
    'employed':    'employed',
    'bachelor':    'bachelor',
    'depchildren': 'depchildren',
    'couple':      'couple',
    'female':      'female',
    'health':      'health',
    'city':        'city'
}
results_df['Variable'] = results_df['Variable'].map(display_names)

# Significance stars
def sig_stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''
results_df['Sig.'] = results_df['p-value'].apply(sig_stars)

# Reorder columns
results_df = results_df[['Variable', 'Coefficient', 'Std. Error',
                         't-statistic', 'p-value',
                         '95% CI Lower', '95% CI Upper', 'Sig.']]

# Colour the p-value column: green if significant at 1%, amber at 10%, grey otherwise
def colour_pvalue(val):
    try:
        v = float(val)
        if v < 0.01:  return 'color: #2b7a2b; font-weight: 700;'
        if v < 0.10:  return 'color: #c98a2c; font-weight: 700;'
        return 'color: #9a9a9a;'
    except:
        return ''

styled_results = (
    results_df.style
        .format({
            'Coefficient':   '{:+.4f}',
            'Std. Error':    '{:.4f}',
            't-statistic':   '{:+.3f}',
            'p-value':       '{:.4f}',
            '95% CI Lower':  '{:+.4f}',
            '95% CI Upper':  '{:+.4f}',
        })
        .map(colour_pvalue, subset=['p-value'])
        .hide(axis='index')
        .set_caption('Table 2 — Linear Probability Model: Determinants of PHI Uptake '
                     '(n = 1,705; HC1 robust standard errors)')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '13px'),
                       ('font-weight', '700'),
                       ('color', '#1f4e79'),
                       ('text-align', 'left'),
                       ('padding', '10px 0'),
                       ('caption-side', 'top')]},
            {'selector': 'th',
             'props': [('background-color', '#1f4e79'),
                       ('color', 'white'),
                       ('font-weight', '600'),
                       ('text-align', 'center'),
                       ('padding', '10px')]},
            {'selector': 'td',
             'props': [('text-align', 'right'),
                       ('padding', '8px 14px')]},
            {'selector': 'td:first-child',
             'props': [('text-align', 'left'),
                       ('font-weight', '600')]},
            {'selector': 'tr:nth-child(even)',
             'props': [('background-color', '#f5f5f5')]}
        ])
)

styled_results

Variable,Coefficient,Std. Error,t-statistic,p-value,95% CI Lower,95% CI Upper,Sig.
Intercept (β₀),-0.0763,0.1099,-0.694,0.4876,-0.2917,+0.1391,
HHincome,+0.0022,0.0002,+11.071,0.0000,+0.0018,+0.0026,***
age,+0.0074,0.0041,+1.796,0.0726,-0.0007,+0.0155,*
age_sq,-0.0000,0.0000,-0.006,0.9950,-0.0001,+0.0001,
employed,+0.0679,0.0282,+2.407,0.0161,+0.0126,+0.1232,**
bachelor,+0.1772,0.0253,+7.012,0.0000,+0.1276,+0.2267,***
depchildren,-0.0310,0.0122,-2.547,0.0109,-0.0549,-0.0071,**
couple,+0.0472,0.0251,+1.877,0.0605,-0.0021,+0.0964,*
female,+0.0407,0.0220,+1.852,0.0641,-0.0024,+0.0838,*
health,-0.0647,0.0117,-5.537,0.0000,-0.0876,-0.0418,***


In [9]:
# === Model Fit Statistics Table — clean academic style ===

fit_stats = pd.DataFrame({
    'Statistic': [
        'Observations',
        'R-squared',
        'Adjusted R-squared',
        'F-statistic',
        'F-statistic p-value',
        'Standard error type'
    ],
    'Value': [
        f'{int(lpm.nobs):,}',
        f'{lpm.rsquared:.4f}',
        f'{lpm.rsquared_adj:.4f}',
        f'{lpm.fvalue:.2f}',
        f'{lpm.f_pvalue:.2e}',
        'HC1 (heteroskedasticity-robust)'
    ]
})

styled_fit = (
    fit_stats.style
        .hide(axis='index')
        .set_caption('Table 3 — Model Fit Statistics')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '13px'),
                       ('font-weight', '700'),
                       ('color', '#1f4e79'),
                       ('text-align', 'left'),
                       ('padding', '10px 0'),
                       ('caption-side', 'top')]},
            {'selector': 'th',
             'props': [('background-color', '#1f4e79'),
                       ('color', 'white'),
                       ('font-weight', '600'),
                       ('text-align', 'center'),
                       ('padding', '10px')]},
            {'selector': 'td',
             'props': [('text-align', 'left'),
                       ('padding', '8px 14px')]},
            {'selector': 'td:first-child',
             'props': [('font-weight', '600')]},
            {'selector': 'tr:nth-child(even)',
             'props': [('background-color', '#f5f5f5')]}
        ])
)

styled_fit

Statistic,Value
Observations,"1,705"
R-squared,0.2087
Adjusted R-squared,0.2040
F-statistic,64.91
F-statistic p-value,6.24e-112
Standard error type,HC1 (heteroskedasticity-robust)


### 2.3 Variable-by-Variable Interpretation

The table below summarises the rationale for including each explanatory variable
in the model, the expected and actual direction of its effect on PHI uptake, the
magnitude of the estimated coefficient, and its statistical significance at the
10% level. The interpretation draws on the economic theory of health-insurance
demand and the empirical literature on the Australian PHI market.

Because this is a Linear Probability Model, each coefficient represents the
estimated change in the **probability** of holding PHI associated with a one-unit
increase in the corresponding variable (or, for binary variables, the difference
between the two categories), holding all other variables constant.

## Section 2.4 — Predicted Probabilities for a Synthetic Individual

This section uses the estimated Linear Probability Model to predict the
probability of holding PHI for a "synthetic average individual" — a
hypothetical person whose characteristics equal the sample means of each
explanatory variable. The predicted probability is then recomputed under
three counterfactual scenarios specified in the assessment brief:
(i) household income doubles, (ii) age increases by 10 years, and (iii)
the individual obtains a bachelor's degree.

In [10]:
# === Step 1: Build the synthetic average individual ===

# Variables in the model (must be in the SAME order as X_vars)
X_vars = ['HHincome', 'age', 'age_sq',
          'employed', 'bachelor', 'depchildren',
          'couple', 'female', 'health', 'city']

# Compute mean of each variable
avg_individual = phi_ind[X_vars].mean().round(2)
print("Synthetic average individual — characteristics:")
print(avg_individual)

Synthetic average individual — characteristics:
HHincome       120.9400
age             53.0900
age_sq        3072.0500
employed         0.6200
bachelor         0.3000
depchildren      1.6000
couple           0.6600
female           0.5800
health           2.6900
city             0.6500
dtype: float64


In [13]:
# === Step 1: Build the synthetic average individual ===

# Variables in the model (must be in the SAME order as X_vars)
X_vars = ['HHincome', 'age', 'age_sq',
          'employed', 'bachelor', 'depchildren',
          'couple', 'female', 'health', 'city']

# Compute mean of each variable
avg_individual = phi_ind[X_vars].mean().round(2)

# Build a display dataframe with friendlier labels and descriptions
display_labels = {
    'HHincome':    'Household income ($000s)',
    'age':         'Age (years)',
    'age_sq':      'Age squared',
    'employed':    'Employed (share)',
    'bachelor':    "Bachelor's degree (share)",
    'depchildren': 'Dependent children (avg.)',
    'couple':      'In a couple (share)',
    'female':      'Female (share)',
    'health':      'Self-reported health (1=best, 5=worst)',
    'city':        'Lives in major city (share)'
}

avg_df = pd.DataFrame({
    'Variable': [display_labels[v] for v in X_vars],
    'Average value': avg_individual.values
})

styled_avg = (
    avg_df.style
        .format({'Average value': '{:.2f}'})
        .hide(axis='index')
        .set_caption('Table 4 — Synthetic Average Individual: Mean Values of Each Variable')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '13px'),
                       ('font-weight', '700'),
                       ('color', '#1f4e79'),
                       ('text-align', 'left'),
                       ('padding', '10px 0'),
                       ('caption-side', 'top')]},
            {'selector': 'th',
             'props': [('background-color', '#1f4e79'),
                       ('color', 'white'),
                       ('font-weight', '600'),
                       ('text-align', 'center'),
                       ('padding', '10px')]},
            {'selector': 'td',
             'props': [('text-align', 'left'),
                       ('padding', '8px 14px')]},
            {'selector': 'td:nth-child(2)',
             'props': [('text-align', 'right')]},
            {'selector': 'td:first-child',
             'props': [('font-weight', '600')]},
            {'selector': 'tr:nth-child(even)',
             'props': [('background-color', '#f5f5f5')]}
        ])
)

styled_avg

Variable,Average value
Household income ($000s),120.94
Age (years),53.09
Age squared,3072.05
Employed (share),0.62
Bachelor's degree (share),0.30
Dependent children (avg.),1.60
In a couple (share),0.66
Female (share),0.58
"Self-reported health (1=best, 5=worst)",2.69
Lives in major city (share),0.65


In [11]:
# === Step 2: Predict P(PHI) for the average individual ===

# Build the input row (add intercept = 1 at the start, in the same column order as the regression)
import numpy as np

avg_row = np.concatenate([[1.0], avg_individual.values])   # 1.0 = constant
baseline_prob = lpm.predict(avg_row)[0]

print(f"Baseline predicted probability of PHI: {baseline_prob:.4f}")
print(f"  → That is approximately {baseline_prob*100:.1f}%")

Baseline predicted probability of PHI: 0.5182
  → That is approximately 51.8%


In [14]:
# === Step 2: Predict P(PHI) for the average individual ===

import numpy as np

# Build the input row (constant first, then variables in X_vars order)
avg_row = np.concatenate([[1.0], avg_individual.values])
baseline_prob = lpm.predict(avg_row)[0]

# Pretty display
baseline_df = pd.DataFrame({
    'Metric': ['Predicted probability of holding PHI',
               'Equivalent percentage'],
    'Value':  [f'{baseline_prob:.4f}',
               f'{baseline_prob*100:.1f}%']
})

styled_baseline = (
    baseline_df.style
        .hide(axis='index')
        .set_caption('Table 5 — Baseline Predicted Probability for the Synthetic Average Individual')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '13px'),
                       ('font-weight', '700'),
                       ('color', '#1f4e79'),
                       ('text-align', 'left'),
                       ('padding', '10px 0'),
                       ('caption-side', 'top')]},
            {'selector': 'th',
             'props': [('background-color', '#1f4e79'),
                       ('color', 'white'),
                       ('font-weight', '600'),
                       ('text-align', 'center'),
                       ('padding', '10px')]},
            {'selector': 'td',
             'props': [('text-align', 'left'),
                       ('padding', '8px 14px')]},
            {'selector': 'td:nth-child(2)',
             'props': [('text-align', 'right'),
                       ('font-weight', '600'),
                       ('color', '#1f4e79')]},
            {'selector': 'tr:nth-child(even)',
             'props': [('background-color', '#f5f5f5')]}
        ])
)

styled_baseline

Metric,Value
Predicted probability of holding PHI,0.5182
Equivalent percentage,51.8%


In [12]:
# === Step 3: Three counterfactual scenarios ===

# We'll create a copy of the average individual for each scenario,
# change ONE variable, and recompute the predicted probability.

# --- Scenario A: Income doubles ---
scenario_A = avg_individual.copy()
scenario_A['HHincome'] = avg_individual['HHincome'] * 2
row_A = np.concatenate([[1.0], scenario_A.values])
prob_A = lpm.predict(row_A)[0]

# --- Scenario B: 10 years older ---
# Remember: we must also update age_sq when age changes!
scenario_B = avg_individual.copy()
new_age = avg_individual['age'] + 10
scenario_B['age'] = new_age
scenario_B['age_sq'] = new_age ** 2
row_B = np.concatenate([[1.0], scenario_B.values])
prob_B = lpm.predict(row_B)[0]

# --- Scenario C: Obtains a bachelor's degree ---
scenario_C = avg_individual.copy()
scenario_C['bachelor'] = 1
row_C = np.concatenate([[1.0], scenario_C.values])
prob_C = lpm.predict(row_C)[0]

# --- Summary table ---
print(f"{'Scenario':<40} {'P(PHI)':>10} {'Change vs baseline':>22}")
print("-" * 75)
print(f"{'Baseline (average individual)':<40} {baseline_prob:>10.4f} {'—':>22}")
print(f"{'A — Income doubles':<40} {prob_A:>10.4f} {prob_A - baseline_prob:>+22.4f}")
print(f"{'B — 10 years older':<40} {prob_B:>10.4f} {prob_B - baseline_prob:>+22.4f}")
print(f"{'C — Obtains a bachelor degree':<40} {prob_C:>10.4f} {prob_C - baseline_prob:>+22.4f}")

Scenario                                     P(PHI)     Change vs baseline
---------------------------------------------------------------------------
Baseline (average individual)                0.5182                      —
A — Income doubles                           0.7845                +0.2663
B — 10 years older                           0.5920                +0.0738
C — Obtains a bachelor degree                0.6422                +0.1240


In [15]:
# === Step 3: Three counterfactual scenarios ===

# --- Scenario A: Income doubles ---
scenario_A = avg_individual.copy()
scenario_A['HHincome'] = avg_individual['HHincome'] * 2
row_A = np.concatenate([[1.0], scenario_A.values])
prob_A = lpm.predict(row_A)[0]

# --- Scenario B: 10 years older ---
# IMPORTANT: must update age_sq together with age
scenario_B = avg_individual.copy()
new_age = avg_individual['age'] + 10
scenario_B['age'] = new_age
scenario_B['age_sq'] = new_age ** 2
row_B = np.concatenate([[1.0], scenario_B.values])
prob_B = lpm.predict(row_B)[0]

# --- Scenario C: Obtains a bachelor's degree ---
scenario_C = avg_individual.copy()
scenario_C['bachelor'] = 1
row_C = np.concatenate([[1.0], scenario_C.values])
prob_C = lpm.predict(row_C)[0]

# === Build summary dataframe ===
scenarios_df = pd.DataFrame({
    'Scenario': [
        'Baseline (average individual)',
        'A — Household income doubles',
        'B — 10 years older',
        "C — Obtains a bachelor's degree"
    ],
    'What changes': [
        '—',
        f'HHincome: $121k → $242k',
        f'age: 53 → 63 (age² updated)',
        'bachelor: 0.30 → 1'
    ],
    'P(PHI)': [
        baseline_prob,
        prob_A,
        prob_B,
        prob_C
    ],
    'P(PHI), %': [
        baseline_prob * 100,
        prob_A * 100,
        prob_B * 100,
        prob_C * 100
    ],
    'Δ vs baseline (pp)': [
        0.0,
        (prob_A - baseline_prob) * 100,
        (prob_B - baseline_prob) * 100,
        (prob_C - baseline_prob) * 100
    ]
})

# Colour the Δ column: green if positive, red if negative
def colour_delta(val):
    try:
        v = float(val)
        if v > 0.5:   return 'color: #2b7a2b; font-weight: 700;'
        if v < -0.5:  return 'color: #b13b3b; font-weight: 700;'
        return 'color: #9a9a9a;'
    except:
        return ''

styled_scenarios = (
    scenarios_df.style
        .format({
            'P(PHI)':              '{:.4f}',
            'P(PHI), %':           '{:.1f}%',
            'Δ vs baseline (pp)':  '{:+.2f}'
        })
        .map(colour_delta, subset=['Δ vs baseline (pp)'])
        .hide(axis='index')
        .set_caption('Table 6 — Predicted PHI Probability under Counterfactual Scenarios')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '13px'),
                       ('font-weight', '700'),
                       ('color', '#1f4e79'),
                       ('text-align', 'left'),
                       ('padding', '10px 0'),
                       ('caption-side', 'top')]},
            {'selector': 'th',
             'props': [('background-color', '#1f4e79'),
                       ('color', 'white'),
                       ('font-weight', '600'),
                       ('text-align', 'center'),
                       ('padding', '10px')]},
            {'selector': 'td',
             'props': [('text-align', 'right'),
                       ('padding', '8px 14px')]},
            {'selector': 'td:first-child, td:nth-child(2)',
             'props': [('text-align', 'left')]},
            {'selector': 'td:first-child',
             'props': [('font-weight', '600')]},
            {'selector': 'tr:nth-child(even)',
             'props': [('background-color', '#f5f5f5')]}
        ])
)

styled_scenarios

Scenario,What changes,P(PHI),"P(PHI), %",Δ vs baseline (pp)
Baseline (average individual),—,0.5182,51.8%,+0.00
A — Household income doubles,HHincome: $121k → $242k,0.7845,78.4%,+26.63
B — 10 years older,age: 53 → 63 (age² updated),0.5920,59.2%,+7.38
C — Obtains a bachelor's degree,bachelor: 0.30 → 1,0.6422,64.2%,+12.40
